# GastroLens — Module 1b: Autoencoder-based Unsupervised Abnormality Detection

This notebook trains a **second** Autoencoder — architecturally identical to the main generation model — but using **only normal-tissue images** (`normal-cecum`, `normal-pylorus`, `normal-z-line`).

Because it never sees disease images during training, its reconstruction error should be **low on normal images** (seen during training) and **high on disease images** (never seen) — giving an unsupervised abnormality signal that needs no disease labels at inference time.

This is a separate deliverable from `01_autoencoder.ipynb` — that notebook remains your main all-8-class generation model; this one is purpose-built for the screening experiment.

## 1. Config

In [ ]:
import os

# ---- Paths ----
DATA_DIR = "./data_dir"   # same root as before — contains all 8 class subfolders
MODEL_DIR = "./models/autoencoder_normal_only"
METRICS_DIR = "./metrics/autoencoder_normal_only"
PLOTS_DIR = "./plots/autoencoder_normal_only"

for d in [MODEL_DIR, METRICS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "autoencoder_normal_only_best.pth")
LOSS_LOG_PATH = os.path.join(METRICS_DIR, "training_loss_log.csv")
ABNORMALITY_METRICS_PATH = os.path.join(METRICS_DIR, "abnormality_detection_metrics.json")
LOSS_PLOT_PATH = os.path.join(PLOTS_DIR, "loss_curve.png")
ERROR_DIST_PLOT_PATH = os.path.join(PLOTS_DIR, "reconstruction_error_distribution.png")
ROC_PLOT_PATH = os.path.join(PLOTS_DIR, "roc_curve.png")
FAILURE_EXAMPLES_PLOT_PATH = os.path.join(PLOTS_DIR, "reconstruction_failure_examples.png")

# ---- Which classes count as "normal" (trained on) vs. "disease" (held out, used only for eval) ----
NORMAL_CLASSES = ["normal-cecum", "normal-pylorus", "normal-z-line"]
DISEASE_CLASSES = ["polyps", "esophagitis", "ulcerative-colitis", "dyed-lifted-polyps", "dyed-resection-margins"]

# ---- Hyperparameters (kept identical to 01_autoencoder.ipynb for a fair comparison) ----
IMG_SIZE = 128
BATCH_SIZE = 64
LATENT_DIM = 128
EPOCHS = 100
LEARNING_RATE = 1e-3
VAL_SPLIT = 0.1          # held-out split from the NORMAL images only, for loss tracking during training
SEED = 42

FORCE_RETRAIN = False

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset — split into normal (train) and disease (held-out, eval only)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
class_to_idx = full_dataset.class_to_idx
print("Classes found:", class_to_idx)

normal_idx = {class_to_idx[c] for c in NORMAL_CLASSES if c in class_to_idx}
disease_idx = {class_to_idx[c] for c in DISEASE_CLASSES if c in class_to_idx}

missing_normal = [c for c in NORMAL_CLASSES if c not in class_to_idx]
missing_disease = [c for c in DISEASE_CLASSES if c not in class_to_idx]
if missing_normal or missing_disease:
    print(f"WARNING — class names not found in dataset: normal={missing_normal}, disease={missing_disease}")
    print("Check the exact folder names printed above and update NORMAL_CLASSES / DISEASE_CLASSES if needed.")

# Sample indices belonging to normal vs disease classes
normal_sample_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label in normal_idx]
disease_sample_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label in disease_idx]

print(f"Normal images (training pool): {len(normal_sample_indices)}")
print(f"Disease images (held-out, eval only): {len(disease_sample_indices)}")

normal_dataset = Subset(full_dataset, normal_sample_indices)
disease_dataset = Subset(full_dataset, disease_sample_indices)

# Split normal images into train/val for training the model
val_size = int(len(normal_dataset) * VAL_SPLIT)
train_size = len(normal_dataset) - val_size
normal_train, normal_val = random_split(
    normal_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(normal_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
normal_val_loader = DataLoader(normal_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
disease_loader = DataLoader(disease_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train (normal only): {len(normal_train)} | Val (normal only): {len(normal_val)} | Disease (eval only): {len(disease_dataset)}")

## 3. Model — identical architecture to 01_autoencoder.ipynb

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.fc = nn.Linear(128 * 16 * 16, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 16 * 16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 128, 16, 16)
        return self.deconv(x)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


model = Autoencoder(LATENT_DIM).to(device)
print(f"Autoencoder parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Train on normal images only (skips automatically if checkpoint exists)

In [ ]:
def evaluate_val_loss(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            recon = model(imgs)
            loss = criterion(recon, imgs)
            total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def train_autoencoder():
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = {"epoch": [], "train_loss": [], "val_loss": [], "epoch_time_sec": []}
    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        start = time.time()
        running_loss = 0.0

        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            recon = model(imgs)
            loss = criterion(recon, imgs)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        train_loss = running_loss / len(normal_train)
        val_loss = evaluate_val_loss(model, normal_val_loader, criterion)
        epoch_time = time.time() - start

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["epoch_time_sec"].append(epoch_time)

        print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | {epoch_time:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss,
                "latent_dim": LATENT_DIM,
                "img_size": IMG_SIZE,
                "trained_on_classes": NORMAL_CLASSES,
            }, CHECKPOINT_PATH)

        pd.DataFrame(history).to_csv(LOSS_LOG_PATH, index=False)

    return history


if os.path.exists(CHECKPOINT_PATH) and not FORCE_RETRAIN:
    print(f"Checkpoint found at {CHECKPOINT_PATH} — loading instead of retraining.")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded model from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.5f})")
    history = pd.read_csv(LOSS_LOG_PATH).to_dict(orient="list") if os.path.exists(LOSS_LOG_PATH) else None
else:
    print("Training normal-only Autoencoder from scratch...")
    history = train_autoencoder()
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best checkpoint (epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.5f})")

In [ ]:
if history is not None:
    plt.figure(figsize=(7, 5))
    plt.plot(history["epoch"], history["train_loss"], label="Train Loss (normal only)")
    plt.plot(history["epoch"], history["val_loss"], label="Validation Loss (normal only)")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Normal-Only Autoencoder Training Loss — GastroLens")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=150)
    plt.show()
    print(f"Saved loss curve to {LOSS_PLOT_PATH}")

## 5. Abnormality detection — reconstruction error as a disease signal
Compute per-image reconstruction error (MSE) separately for held-out normal images and disease images. If the model has genuinely learned only "normal," disease images should show visibly higher error.

In [ ]:
def per_image_reconstruction_errors(model, loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            recon = model(imgs)
            # per-image MSE, not averaged across the batch
            batch_errors = torch.mean((recon - imgs) ** 2, dim=[1, 2, 3])
            errors.extend(batch_errors.cpu().numpy().tolist())
    return np.array(errors)


normal_errors = per_image_reconstruction_errors(model, normal_val_loader)
disease_errors = per_image_reconstruction_errors(model, disease_loader)

print(f"Normal images  — mean error: {normal_errors.mean():.5f} | std: {normal_errors.std():.5f} | n={len(normal_errors)}")
print(f"Disease images — mean error: {disease_errors.mean():.5f} | std: {disease_errors.std():.5f} | n={len(disease_errors)}")

In [ ]:
# Distribution plot — normal vs disease reconstruction error
plt.figure(figsize=(7, 5))
plt.hist(normal_errors, bins=40, alpha=0.6, label="Normal (held-out)", color="tab:blue")
plt.hist(disease_errors, bins=40, alpha=0.6, label="Disease (unseen classes)", color="tab:red")
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Number of Images")
plt.title("Reconstruction Error: Normal vs. Disease Images")
plt.legend()
plt.tight_layout()
plt.savefig(ERROR_DIST_PLOT_PATH, dpi=150)
plt.show()
print(f"Saved error distribution plot to {ERROR_DIST_PLOT_PATH}")

In [ ]:
# ROC curve / AUC — treating reconstruction error as an anomaly score
# pip install scikit-learn --break-system-packages   (if not already installed)
from sklearn.metrics import roc_curve, roc_auc_score

y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(disease_errors))])  # 0 = normal, 1 = disease
y_score = np.concatenate([normal_errors, disease_errors])  # higher error = more likely disease

fpr, tpr, thresholds = roc_curve(y_true, y_score)
auc_score = roc_auc_score(y_true, y_score)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_score:.3f})", color="tab:purple")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Abnormality Detection ROC Curve — Reconstruction Error as Anomaly Score")
plt.legend()
plt.tight_layout()
plt.savefig(ROC_PLOT_PATH, dpi=150)
plt.show()

print(f"AUC: {auc_score:.4f}")
print(f"Saved ROC curve to {ROC_PLOT_PATH}")

# Suggest an operating threshold using Youden's J statistic (maximizes TPR - FPR)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]
print(f"Suggested error threshold for flagging abnormality: {best_threshold:.5f}")
print(f"  -> at this threshold: TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f}")

In [ ]:
# Save abnormality detection metrics summary
abnormality_metrics = {
    "trained_on_classes": NORMAL_CLASSES,
    "evaluated_disease_classes": DISEASE_CLASSES,
    "normal_error_mean": float(normal_errors.mean()),
    "normal_error_std": float(normal_errors.std()),
    "disease_error_mean": float(disease_errors.mean()),
    "disease_error_std": float(disease_errors.std()),
    "roc_auc": float(auc_score),
    "suggested_threshold": float(best_threshold),
}

with open(ABNORMALITY_METRICS_PATH, "w") as f:
    json.dump(abnormality_metrics, f, indent=2)

print(f"Saved abnormality detection metrics to {ABNORMALITY_METRICS_PATH}")
print(json.dumps(abnormality_metrics, indent=2))

## 6. Visualize reconstruction failures on disease images
Shows original vs. reconstructed side by side for a few disease images with the highest error — a strong visual for your report/viva.

In [ ]:
# Collect disease images + their errors + reconstructions together so we can pick the worst ones
model.eval()
all_imgs, all_recons, all_errors = [], [], []

with torch.no_grad():
    for imgs, _ in disease_loader:
        imgs = imgs.to(device)
        recon = model(imgs)
        batch_errors = torch.mean((recon - imgs) ** 2, dim=[1, 2, 3])
        all_imgs.append(imgs.cpu())
        all_recons.append(recon.cpu())
        all_errors.append(batch_errors.cpu())

all_imgs = torch.cat(all_imgs)
all_recons = torch.cat(all_recons)
all_errors = torch.cat(all_errors)

# Pick the top-N highest-error (worst reconstructed = most "flagged") disease images
n_show = 6
top_indices = torch.topk(all_errors, n_show).indices

fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4))
for col, idx in enumerate(top_indices):
    axes[0, col].imshow(all_imgs[idx].permute(1, 2, 0).numpy())
    axes[0, col].axis("off")
    axes[0, col].set_title("Original" if col == 0 else "")

    axes[1, col].imshow(all_recons[idx].permute(1, 2, 0).numpy())
    axes[1, col].axis("off")
    axes[1, col].set_title(f"Recon (err={all_errors[idx]:.4f})")

plt.suptitle("Highest-Error Disease Images — Model Trained Only on Normal Tissue")
plt.tight_layout()
plt.savefig(FAILURE_EXAMPLES_PLOT_PATH, dpi=150)
plt.show()
print(f"Saved reconstruction failure examples to {FAILURE_EXAMPLES_PLOT_PATH}")

## Output summary

| Artifact | Path |
|---|---|
| Trained model (normal-only) | `models/autoencoder_normal_only/autoencoder_normal_only_best.pth` |
| Training loss log | `metrics/autoencoder_normal_only/training_loss_log.csv` |
| Abnormality detection metrics (AUC, error stats, threshold) | `metrics/autoencoder_normal_only/abnormality_detection_metrics.json` |
| Loss curve | `plots/autoencoder_normal_only/loss_curve.png` |
| Reconstruction error distribution (normal vs disease) | `plots/autoencoder_normal_only/reconstruction_error_distribution.png` |
| ROC curve | `plots/autoencoder_normal_only/roc_curve.png` |
| Worst-reconstructed disease examples | `plots/autoencoder_normal_only/reconstruction_failure_examples.png` |

**Key result to report:** the ROC AUC score quantifies how well reconstruction error alone separates normal from diseased tissue — this is your unsupervised abnormality detection result, independent of and complementary to the main generation-focused Autoencoder in `01_autoencoder.ipynb`.